# 📈 Quantitative Backtesting Framework: Trend-Following vs. Buy & Hold

## Overview
This notebook contains a professional-grade, object-oriented backtesting engine designed to evaluate and compare algorithmic trading strategies against traditional Buy & Hold approaches. It is built to rigorously stress-test strategies across decades of historical data using dynamic rolling windows to identify average expected performance, median outcomes, and absolute worst-case scenarios.

## Core Capabilities
* **Rolling Window Analysis:** Instead of a single static backtest, the engine runs hundreds of sequential simulations (e.g., rolling 26-year periods stepping forward month-by-month) to eliminate timing luck and reveal how a strategy performs across different market regimes (e.g., Dot-com crash, 2008 Financial Crisis).
* **Unbiased Performance Metrics (TWR):** Utilizes **Time-Weighted Return (TWR)** as the primary grading mechanism. TWR isolates the algorithm's pure compounding performance by neutralizing the distorting effects of external cash flows.
* **Dollar Cost Averaging (DCA):** Fully supports periodic capital injections, calculating both the strategy's internal efficiency (TWR) and the investor's actual wallet growth (Total ROI).
* **Leverage & Decay Simulation:** Accurately models the mathematical drag and volatility decay inherent in leveraged ETFs (e.g., 2x or 3x exposure) over long time horizons.
* **Worst-Case Isolation:** Automatically flags the exact historical start dates that produced the maximum drawdowns and worst returns for targeted post-mortem analysis.

## Architecture
The framework relies on a modular, Object-Oriented Programming (OOP) design:
1. `BaseStrategy` **(The Logic):** Defines the trading rules and signal generation (e.g., SMA 200 with ATR buffers).
2. `Backtester` **(The Math Engine):** Handles the daily accounting, cash flows, order execution, equity curve generation, and metric calculation (TWR, Max DD) for a single time period.
3. `RollingBacktester` **(The Orchestrator):** Manages bulk experiment execution, dynamically generating valid start/end dates, aggregating results, and outputting summary statistics.

---
*Note: Ensure all dependencies (Pandas, NumPy, yfinance) are installed and imported before running the engine cells.*


In [1]:
from strat_backtest import *

In [2]:
# 1. Create environment
env = Backtester(
    base_ticker="^NDX",
    leverage=3,
    initial_fund=10000,
    annual_dca=0,
    verbose=True
)

# 2. Define strategies (The "Players")
strat_hold = BuyAndHold()
strat_sma_atr = SMATrendFollowing(sma_window=200, atr_multiplier=2.5)

# 3. Run them!
results_1 = env.run(strat_hold)
results_2 = env.run(strat_sma_atr)

--- Running Buy & Hold: 3x ^NDX | Lump Sum ---
Final Value:    $4,883.42
Lowest Value:   $15.24
Strategy TWR:   -2.83%
Max Drawdown:   -99.9828%
Total Trades:   0
Avg Cash Hold:  0.0 Trading Days | Total Cash Periods: 0


--- Running SMA 200 - ATR Buffer (x2.5): 3x ^NDX | Lump Sum ---
Final Value:    $769,075.50
Lowest Value:   $9,617.14
Strategy TWR:   18.97%
Max Drawdown:   -82.1059%
Total Trades:   15
Avg Cash Hold:  118.1 Trading Days | Total Cash Periods: 15




In [4]:
# 1. Define Dates
period_years = 26
end_date = pd.Timestamp.today() - pd.DateOffset(years=period_years)
monthly_start_dates = pd.date_range(
    start='1980-01-15',
    end=end_date,
    freq=pd.DateOffset(months=1)
)

# 2. Define Environments (Leverage Configs)
my_configs = [
    {"name": "3x Leverage", "leverage": 3, "expense": 0.0095},
    {"name": "2x Leverage", "leverage": 2, "expense": 0.0095},
    {"name": "1x Leverage", "leverage": 1, "expense": 0.0020}
]

# 3. Define Strategies
strategies = [
    BuyAndHold(),
    SMATrendFollowing(sma_window=200, atr_multiplier=2.5),
    VolatilityFilter(name="VIX < 25", vix_threshold=25),
    EMACrossover(name="EMA 50/200"),
    RSIMeanReversion(name="RSI 30/70")
]

# 4. Run the Suite!
results_dict = run_experiment_suite(
    configs=my_configs,
    strategies=strategies,
    start_dates=monthly_start_dates,
    period_years=26,
    initial_fund=10000,
    annual_dca=10000
)

🚀 RUNNING MONTHLY ROLLING BACKTEST: 3x Leverage...
🚀 RUNNING MONTHLY ROLLING BACKTEST: 2x Leverage...
🚀 RUNNING MONTHLY ROLLING BACKTEST: 1x Leverage...

📊 SUMMARY STATISTICS (TWR & Drawdowns)
--- 3x Leverage ---
[Buy & Hold]
  TWR       -> Avg:     2.01% | Med:     1.85%
              Worst:    -7.44% (Started: 2000-03-15)
  Max DD:   -99.97% (Started: 1982-12-15)

[SMA 200 - ATR Buffer (x2.5)]
  TWR       -> Avg:    21.40% | Med:    21.99%
              Worst:    12.02% (Started: 2000-03-15)
  Max DD:   -81.89% (Started: 1983-07-15)

[VIX < 25]
  TWR       -> Avg:    -3.02% | Med:    -2.84%
              Worst:    -9.91% (Started: 2000-02-15)
  Max DD:   -99.85% (Started: 1980-01-15)

[EMA 50/200]
  TWR       -> Avg:    18.97% | Med:    20.54%
              Worst:     9.32% (Started: 1987-02-15)
  Max DD:   -91.31% (Started: 1985-10-15)

[RSI 30/70]
  TWR       -> Avg:    -6.81% | Med:    -6.65%
              Worst:   -14.74% (Started: 1983-03-15)
  Max DD:   -99.53% (Started: 1985-1

In [5]:
# 1. Define Dates
period_years = 26
end_date = pd.Timestamp.today() - pd.DateOffset(years=period_years)
monthly_start_dates = pd.date_range(
    start='1980-01-15',
    end=end_date,
    freq=pd.DateOffset(months=1)
)

# 2. Define Environments (Leverage Configs)
my_configs = [
    {"name": "3x Leverage", "leverage": 3, "expense": 0.0095},
    {"name": "2x Leverage", "leverage": 2, "expense": 0.0095},
    {"name": "1x Leverage", "leverage": 1, "expense": 0.0020}
]

# 3. Define Strategies
strategies = [
    BuyAndHold(),
    SMATrendFollowing(sma_window=200, atr_multiplier=2.5),
    VolatilityFilter(name="VIX < 25", vix_threshold=25),
    EMACrossover(name="EMA 50/200"),
    RSIMeanReversion(name="RSI 30/70")
]

# 4. Run the Suite!
results_dict = run_experiment_suite(
    configs=my_configs,
    strategies=strategies,
    start_dates=monthly_start_dates,
    period_years=26,
    initial_fund=10000,
    annual_dca=0
)

🚀 RUNNING MONTHLY ROLLING BACKTEST: 3x Leverage...
🚀 RUNNING MONTHLY ROLLING BACKTEST: 2x Leverage...
🚀 RUNNING MONTHLY ROLLING BACKTEST: 1x Leverage...

📊 SUMMARY STATISTICS (TWR & Drawdowns)
--- 3x Leverage ---
[Buy & Hold]
  TWR       -> Avg:     2.01% | Med:     1.85%
              Worst:    -7.44% (Started: 2000-03-15)
  Max DD:   -99.98% (Started: 1985-10-15)

[SMA 200 - ATR Buffer (x2.5)]
  TWR       -> Avg:    21.40% | Med:    21.99%
              Worst:    12.02% (Started: 2000-03-15)
  Max DD:   -82.11% (Started: 1989-09-15)

[VIX < 25]
  TWR       -> Avg:    -3.02% | Med:    -2.84%
              Worst:    -9.91% (Started: 2000-02-15)
  Max DD:   -99.89% (Started: 1988-11-15)

[EMA 50/200]
  TWR       -> Avg:    18.97% | Med:    20.54%
              Worst:     9.32% (Started: 1987-02-15)
  Max DD:   -92.50% (Started: 1987-03-15)

[RSI 30/70]
  TWR       -> Avg:    -6.81% | Med:    -6.65%
              Worst:   -14.74% (Started: 1983-03-15)
  Max DD:   -99.84% (Started: 1986-0

In [13]:
# 1. Create environment
env = Backtester(
    base_ticker="^NDX",
    leverage=3,
    initial_fund=10000,
    annual_dca=0,
    verbose=True
)

# 2. Define strategies (The "Players")
strat_VIX = VolatilityFilter(name="VIX < 25", vix_threshold=25)

# 3. Run them!
results_1 = env.run(strat_VIX)

--- Running VIX < 25: 3x ^NDX | Lump Sum ---
Final Value:    $1,394.66
Lowest Value:   $47.91
Strategy TWR:   -7.58%
Max Drawdown:   -99.8890%
Total Trades:   137
Avg Cash Hold:  9.7 Trading Days | Total Cash Periods: 137


